# Vietnamese Generative VQA — Colab Demo

Notebook này chạy ứng dụng Gradio từ thư mục `MyDrive/TtI/Vi-VQA-End-to-End`. Mã nguồn và adapter được chép vào `/content` để giảm độ trễ; các base model lớn vẫn được dùng từ Drive. Hãy chọn GPU T4 trước khi chạy toàn bộ notebook.

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True, timeout_ms=600000)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Chưa có GPU. Chọn Runtime > Change runtime type > T4 GPU rồi chạy lại.")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import os
import shutil

DRIVE_PROJECT = Path("/content/drive/MyDrive/TtI/Vi-VQA-End-to-End")
LOCAL_PROJECT = Path("/content/Vi-VQA-End-to-End")
if not DRIVE_PROJECT.is_dir():
    raise FileNotFoundError(f"Không tìm thấy thư mục Drive: {DRIVE_PROJECT}")

# Luôn rời thư mục project cũ trước khi xoá để notebook có thể chạy lại nhiều lần.
os.chdir("/content")
if LOCAL_PROJECT.exists():
    shutil.rmtree(LOCAL_PROJECT)
LOCAL_PROJECT.mkdir(parents=True)

# Chỉ chép mã nguồn nhỏ; không chép lại các base model 20+ GB.
for name in ["app.py", "requirements-app.txt", "requirements.txt", "README.md", "src", "scripts", "configs", "assets"]:
    source = DRIVE_PROJECT / name
    if not source.exists():
        continue
    target = LOCAL_PROJECT / name
    if source.is_dir():
        shutil.copytree(source, target)
    else:
        shutil.copy2(source, target)

# Adapter nhỏ hơn base model và được chép vào ổ Colab để nạp nhanh hơn.
for relative in ["outputs-2B-distilled/final_lora", "outputs-7B/final_lora"]:
    source = DRIVE_PROJECT / relative
    target = LOCAL_PROJECT / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, target)

# Dùng base model đã lưu trên Drive mà không nhân đôi dung lượng.
(LOCAL_PROJECT / "models").symlink_to(DRIVE_PROJECT / "models", target_is_directory=True)
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"
print("Đã chuẩn bị project tại:", LOCAL_PROJECT)

In [ ]:
%cd /content/Vi-VQA-End-to-End
!pip install -q -r requirements-app.txt

In [ ]:
!python app.py --check

In [ ]:
# Cell này tiếp tục chạy để giữ server và tunnel hoạt động.
# Mở đường dẫn DEMO_URL được in đậm ở đầu ra.
!python -u scripts/run_colab_demo.py